# SHViT + Food-101 on Google Colab

This notebook walks through:
1. Enabling GPU and checking the environment
2. Cloning SHViT and installing dependencies
3. Downloading pretrained SHViT-S4 weights
4. Downloading Food-101 and organizing it into ImageNet-style layout
5. Verifying the model loads and runs inference
6. Running SHViT's official eval script

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## 0. Check GPU & environment

In [ ]:
import torch

print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

import sys
print('Python version  :', sys.version.split()[0])

## 1. (Optional) Mount Google Drive

Food-101 is ~5 GB and **Colab storage is wiped when the session ends**.
Mounting Drive lets you keep the dataset across sessions.
Skip this cell if you are happy to re-download each time.

In [ ]:
USE_DRIVE = False   # <-- set True to persist data in Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/food101'
else:
    DATA_ROOT = '/content/data/food101'

print('Dataset will be stored at:', DATA_ROOT)

## 2. Clone SHViT

In [ ]:
import os

if not os.path.isdir('/content/SHViT'):
    !git clone https://github.com/ysj9909/SHViT.git /content/SHViT
else:
    print('SHViT already cloned, skipping.')

!ls /content/SHViT

## 3. Install dependencies

Colab ships with PyTorch 2.x which satisfies SHViT's `>=1.11` requirement,
so we only need to install the extra packages from `requirements.txt`.

`--no-deps` on timm avoids overwriting Colab's torch/torchvision with
the older versions timm 0.5.4 would otherwise pull in.

In [ ]:
!pip install -q timm==0.5.4 --no-deps
!pip install -q einops==0.4.1 fvcore easydict yacs scikit-image==0.19.3 onnx
print('All dependencies installed.')

## 4. Download SHViT-S4 pretrained weights

In [ ]:
WEIGHTS_DIR = '/content/weights'
WEIGHTS_PATH = f'{WEIGHTS_DIR}/shvit_s4.pth'

os.makedirs(WEIGHTS_DIR, exist_ok=True)

if not os.path.exists(WEIGHTS_PATH):
    !wget -q --show-progress \
        https://github.com/ysj9909/SHViT/releases/download/v1.0/shvit_s4.pth \
        -O {WEIGHTS_PATH}
else:
    print('Weights already downloaded, skipping.')

size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
print(f'Checkpoint size: {size_mb:.1f} MB  ->  {WEIGHTS_PATH}')

## 5. Download Food-101 and organize into ImageNet-style layout

This writes `prepare_food101.py` from the repo into Colab and runs it.
The script uses `torchvision.datasets.Food101` to download the dataset
and then hard-links images into `train/<class>/` and `val/<class>/` folders.

In [ ]:
# Write prepare_food101.py inline so the notebook is self-contained
script = '''
import argparse, shutil
from pathlib import Path
import torchvision.datasets as datasets

def download_food101(root):
    print(f"Downloading Food-101 train split to {root} ...")
    datasets.Food101(root=str(root), split="train", download=True)
    print(f"Downloading Food-101 test split to {root} ...")
    datasets.Food101(root=str(root), split="test", download=True)

def reorganize(root):
    food_root = root / "food-101"
    images_dir = food_root / "images"
    meta_dir   = food_root / "meta"
    split_map  = {"train": "train.txt", "val": "test.txt"}

    for split_name, meta_file in split_map.items():
        split_dir = root / split_name
        lines = (meta_dir / meta_file).read_text().strip().splitlines()
        print(f"Organizing {split_name} split ({len(lines)} images) ...")
        for line in lines:
            class_name, img_id = line.split("/")
            src = images_dir / class_name / f"{img_id}.jpg"
            dst_dir = split_dir / class_name
            dst_dir.mkdir(parents=True, exist_ok=True)
            dst = dst_dir / f"{img_id}.jpg"
            if not dst.exists():
                try:
                    dst.hardlink_to(src)
                except (AttributeError, OSError):
                    shutil.copy2(src, dst)
        n_cls = len(list(split_dir.iterdir()))
        n_img = sum(1 for _ in split_dir.rglob("*.jpg"))
        print(f"  {split_name}: {n_cls} classes, {n_img} images -> {split_dir}")

parser = argparse.ArgumentParser()
parser.add_argument("--root", type=Path, default=Path("data/food101"))
args = parser.parse_args()
args.root.mkdir(parents=True, exist_ok=True)
download_food101(args.root)
reorganize(args.root)
print("Done! Pass --data-path", args.root, "to SHViT main.py")
'''

with open('/content/prepare_food101.py', 'w') as f:
    f.write(script)

!python /content/prepare_food101.py --root {DATA_ROOT}

In [ ]:
# Sanity check: count images per split
import pathlib
for split in ('train', 'val'):
    p = pathlib.Path(DATA_ROOT) / split
    classes = list(p.iterdir())
    images  = list(p.rglob('*.jpg'))
    print(f'{split:5s}  {len(classes):3d} classes  {len(images):6d} images')

## 6. Verify model loads and runs inference

Loads the SHViT-S4 checkpoint and runs 50 val images through it.
Predictions are ImageNet class indices (not Food-101 labels) — accuracy
will be low until the model is fine-tuned. The goal here is just to
confirm no import / shape errors occur.

In [ ]:
import sys, time, pathlib
import torch
from PIL import Image
from torchvision import transforms

# Add SHViT to path
sys.path.insert(0, '/content/SHViT')

from models import SHViT  # registers SHViT_* with timm
import timm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

# Build model
model = timm.create_model('SHViT_S4', pretrained=False, num_classes=1000)

# Load checkpoint
ckpt = torch.load(WEIGHTS_PATH, map_location='cpu')
state_dict = ckpt.get('model', ckpt)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f'Missing keys: {len(missing)}   Unexpected keys: {len(unexpected)}')

model.to(DEVICE).eval()
print('Model loaded successfully.')

In [ ]:
NUM_IMAGES = 50

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_dir = pathlib.Path(DATA_ROOT) / 'val'
items = []
for cls_dir in sorted(val_dir.iterdir()):
    for img_path in cls_dir.glob('*.jpg'):
        items.append((img_path, cls_dir.name))
        if len(items) >= NUM_IMAGES:
            break
    if len(items) >= NUM_IMAGES:
        break

print(f'Running inference on {len(items)} images ...')
t0 = time.perf_counter()
results = []
with torch.no_grad():
    for img_path, true_class in items:
        x = transform(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
        pred = int(model(x).argmax(1).item())
        results.append((img_path.name, true_class, pred))

elapsed = time.perf_counter() - t0
print(f'\n{"Image":<30} {"True class":<25} {"Pred idx":>8}')
print('-' * 65)
for name, cls, pred in results[:15]:
    print(f'{name:<30} {cls:<25} {pred:>8}')
print(f'\nTotal: {elapsed:.2f}s  ({elapsed/len(results)*1000:.1f} ms/image)')
print('\n[OK] Model ran without errors.')

## 7. Run SHViT's official eval script

Uses `--data-set IMNET` which accepts any ImageNet-style folder layout.
The `--eval` flag skips training entirely.

> Expected output: accuracy numbers relative to **ImageNet-1K classes**.
> Fine-tune the model on Food-101 to get meaningful food accuracy.

In [ ]:
!python /content/SHViT/main.py \
    --model SHViT_S4 \
    --eval \
    --resume {WEIGHTS_PATH} \
    --data-path {DATA_ROOT} \
    --data-set IMNET \
    --batch-size 64 \
    --num_workers 2 \
    --device cuda

## Next steps — fine-tuning on Food-101

To actually train SHViT on Food-101, replace `--eval` with full training args.
Key changes from ImageNet defaults:
- `--nb_classes 101` — Food-101 has 101 classes, not 1000
- `--finetune` instead of `--resume` when starting from ImageNet weights
- Reduce `--epochs` (e.g. 30–50 for fine-tuning)
- Lower `--lr` (e.g. 1e-4)

```bash
!python /content/SHViT/main.py \\
    --model SHViT_S4 \\
    --finetune {WEIGHTS_PATH} \\
    --data-path {DATA_ROOT} \\
    --data-set IMNET \\
    --nb_classes 101 \\
    --batch-size 64 \\
    --epochs 30 \\
    --lr 1e-4 \\
    --output_dir /content/output
```